In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

import warnings
warnings.filterwarnings('ignore')


# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
ai_path = os.path.join(path, 'Q1_data.csv')
df_ai = pd.read_csv(ai_path)

print(f"Dataset shape: {df_ai.shape}")
df_ai.head()

In [ ]:
df_ai.info()

In [ ]:
df_ai.describe()

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(df_ai['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('delivery_time')
plt.xlabel('time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
cols = ['Order_ID',	'Distance_km',	'Preparation_Time_min',	'Courier_Experience_yrs',	'Delivery_Time']
df_clean = df_ai[cols].copy()
print(f"Before: {df_clean.shape}")
df_clean = df_clean.drop(columns=['Order_ID'])
print(f"After dropping missing Order_ID: {df_clean.shape}")

In [ ]:

missing_percentage = (df_ai.isnull().sum() / len(df_ai)) * 100
missing_data = pd.DataFrame({
    'Column': missing_percentage.index,
    'Missing_Percentage': missing_percentage.values
})
missing_data = missing_data[missing_data['Missing_Percentage'] > 0].sort_values('Missing_Percentage', ascending=False)

print("Missing Data Analysis:")
missing_data.head(10)

In [ ]:
print("Checking for duplicate rows...")
# التحقق من وجود صفوف مكررة


In [ ]:
# Encode categorical columns - converts text to integers
categorical_cols = ['Distance_km',	'Preparation_Time_min',	'Courier_Experience_yrs',	'Delivery_Time']
for col in categorical_cols:
    le = LabelEncoder()
    df_clean[col] = le.fit_transform(df_clean[col].astype(str))

df_clean.head()

In [ ]:
scaler = StandardScaler()
df_clean['Distance_km'] = df_clean['Courier_Experience_yrs']

# Mileage per year: high usage = more wear
df_clean['miles_per_yrs'] = df_clean['Delivery_Time'] / (df_clean['Distance_km'])


# Interaction: captures combined effect of age and annual mileage
df_clean['age_x_miles_per_yrs'] = df_clean['miles_per_yrs'] * df_clean['age']

# Quick preview of engineered features
df_clean[['Distance_km',	'Preparation_Time_min',	'Courier_Experience_yrs',	'Delivery_Time']].head()

In [ ]:
# not needed

In [ ]:
# Define features (X) and target (y)
feature_cols = ['Distance_km',	'Preparation_Time_min',	'Courier_Experience_yrs',	'Delivery_Time']
X = df_clean[feature_cols]
y = df_clean['Distance_km']

# Train-test split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

In [ ]:
# Scale features - fit on train, transform both
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\nScaled ranges - Min: {X_train_scaled.min():.2f}, Max: {X_train_scaled.max():.2f}")
pd.DataFrame(X_train_scaled, columns=X_train.columns).head(3)


In [ ]:
model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
model.fit(X_train_scaled, y_train)
print("Model trained!")

In [ ]:
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []
rmse_scores = []

for train_idx, val_idx in kfold.split(X_train_scaled):
    X_fold_train, X_fold_val = X_train_scaled[train_idx], X_train_scaled[val_idx]
    y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    # Train and predict
    model.fit(X_fold_train, y_fold_train)
    y_fold_pred = model.predict(X_fold_val)

    # Calculate metrics
    mae_scores.append(mean_absolute_error(y_fold_val, y_fold_pred))
    rmse_scores.append(np.sqrt(mean_squared_error(y_fold_val, y_fold_pred)))

mae_scores = np.array(mae_scores)
rmse_scores = np.array(rmse_scores)

print(f"5-Fold CV Results:")
print(f"MAE:  ${mae_scores.mean():,.2f}")
print(f"RMSE: ${rmse_scores.mean():,.2f}")

In [ ]:
# Feature importance
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Predict and evaluate
y_pred = model.predict(X_test_scaled)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"MAE:  ${mae:,.2f}")
print(f"RMSE: ${rmse:,.2f}")

In [ ]:
# Task Bonus: Write your code here: